# Curvature / Importance of ICL — Kaggle T4 Runner

**Phase 1** (cells 1–6): train + diagnostic + kill-test. COMPLETE.

**Phase A** (cells 7–11): K/D regime audit + LOO stability + GATE A.

**Phase B** (cells 12+): only if GATE A = GO.

In [ ]:
# Cell 1: Environment setup
import subprocess, sys, os

REPO_URL = 'https://github.com/bilal-ahmed-khan7412/curvature-icl.git'
REPO_DIR = '/kaggle/working/curvature-icl'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2: Config + paths
from src.utils import load_config, ensure_dir

cfg = load_config('configs/default.yaml')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg['paths']['checkpoint_dir'] = '/kaggle/working/results/checkpoints'
cfg['paths']['metrics_dir']    = '/kaggle/working/results/metrics'
cfg['paths']['figures_dir']    = '/kaggle/working/results/figures'

for p in cfg['paths'].values():
    ensure_dir(p)

print('Config loaded. Device:', DEVICE)

In [ ]:
# Cell 3: Phase 1 — Train (or load existing checkpoint)
SKIP_TRAIN = True  # Phase 1 already done — load checkpoint

from src.toy_icl import build_model
from src.utils import load_latest_checkpoint

if SKIP_TRAIN:
    model = build_model(cfg, DEVICE)
    step = load_latest_checkpoint(cfg['paths']['checkpoint_dir'], model, device=DEVICE)
    assert step > 0, 'No checkpoint found — run with SKIP_TRAIN=False first'
    print(f'Loaded checkpoint at step {step}')
else:
    from src.train_toy import train
    model = train(cfg=cfg, device=DEVICE)

model.eval()
print('Model ready.')

In [ ]:
# Cell 4: Soft gate — confirm ICL formed (from Phase 1; skip if already done)
import numpy as np
from scipy.stats import pearsonr
from src.toy_icl import sample_tasks, build_token_sequence, ols_predict
from src.utils import set_seed

set_seed(cfg['seed'])
rng = np.random.default_rng(cfg['seed'])
task_cfg = cfg['task']
batch = sample_tasks(256, task_cfg['context_len'], task_cfg['input_dim'],
                      task_cfg['noise_std'], rng, DEVICE)

with torch.no_grad():
    seq = build_token_sequence(batch.xs, batch.ys, batch.x_query)
    preds = model(seq).cpu().numpy()
targets   = batch.y_query.cpu().numpy()
ols_preds = ols_predict(batch.xs, batch.ys, batch.x_query).cpu().numpy()

ols_corr, _ = pearsonr(preds, ols_preds)
print(f'Model-OLS Pearson: {ols_corr:.4f}')
assert ols_corr > 0.8, f'ICL not formed (corr={ols_corr:.3f})'

In [ ]:
# Cell 5–6: Phase 1 figures (already done in prior run — skip)
print('Phase 1 complete. Proceeding to Phase A.')

---
## Phase A — Audit the Ground Truth

**Goal:** Find the K/D regime where LOO importance is non-degenerate and stable. Re-test the 3 original methods there. Report GATE A.

In [ ]:
# Cell 7: A.1 — K/D Regime Sweep
# Expected runtime: ~15–25 min on T4
from src.phase_a import sweep_kd_regimes, K_VALUES
from src.viz_phase_a import plot_regime_sweep, plot_importance_distribution
from src.utils import sync_to_kaggle_output

print(f'K values to sweep: {K_VALUES}')
print(f'Original K={cfg["task"]["context_len"]}, D={cfg["task"]["input_dim"]} → K/D={cfg["task"]["context_len"]/cfg["task"]["input_dim"]:.1f}')

sweep = sweep_kd_regimes(model, cfg, DEVICE, seed=cfg['seed'])

p = plot_regime_sweep(sweep, fig_dir=cfg['paths']['figures_dir'])
sync_to_kaggle_output(p, 'figures')

for k_plot in [8, 16, 32]:
    if k_plot in sweep:
        p = plot_importance_distribution(sweep[k_plot], fig_dir=cfg['paths']['figures_dir'])
        sync_to_kaggle_output(p, 'figures')

print('\nRegime sweep summary:')
print(f'{"K":>5} {"K/D":>6} {"Mean imp":>10} {"Within-range":>14}')
for k in sorted(sweep.keys()):
    r = sweep[k]
    orig = ' ← ORIGINAL' if k == cfg['task']['context_len'] else ''
    print(f'{k:>5} {r["kd_ratio"]:>6.2f} {r["mean_importance"]:>10.4f} {r["within_task_range_mean"]:>14.4f}{orig}')

In [ ]:
# Cell 8: Pick candidate good regime (highest within-task importance spread)
best_k_sweep = max(sweep.keys(), key=lambda k: sweep[k]['within_task_range_mean'])
print(f'Candidate best K: {best_k_sweep}  (K/D={best_k_sweep/cfg["task"]["input_dim"]:.2f})')
print(f'Mean importance: {sweep[best_k_sweep]["mean_importance"]:.4f}')
print(f'Within-task range: {sweep[best_k_sweep]["within_task_range_mean"]:.4f}')

In [ ]:
# Cell 9: A.2 — LOO Stability Audit at best K
# Expected runtime: ~5–10 min
from src.phase_a import loo_stability_audit
from src.viz_phase_a import plot_stability_audit

stability = loo_stability_audit(model, cfg, k=best_k_sweep, device=DEVICE, seed=cfg['seed'])

p = plot_stability_audit(stability, sweep, fig_dir=cfg['paths']['figures_dir'])
sync_to_kaggle_output(p, 'figures')

print(f'Noise-seed CV:      {stability["noise_seed_cv"]:.4f}  ({"STABLE" if stability["noise_seed_cv"] < 0.5 else "UNSTABLE"})')
print(f'Query CV:           {stability["query_cv"]:.4f}')
print(f'Context-resample CV:{stability["context_resample_cv"]:.4f}')
gt_def = 'loo' if stability['is_stable'] else 'shapley'
print(f'Ground truth choice: {gt_def}')

In [ ]:
# Cell 10: A.3 — Re-test 3 original methods in good regime
# Expected runtime: ~5–20 min depending on gt_def and K
from src.phase_a import retest_methods, gate_a_verdict
from src.viz_phase_a import plot_retest_comparison, plot_gate_a_summary
from src.utils import save_metrics

retest = retest_methods(model, cfg, k=best_k_sweep, ground_truth=gt_def,
                         device=DEVICE, seed=cfg['seed'])

p = plot_retest_comparison(retest, fig_dir=cfg['paths']['figures_dir'])
sync_to_kaggle_output(p, 'figures')

verdict = gate_a_verdict(sweep, stability)
p = plot_gate_a_summary(sweep, stability, retest, verdict,
                         fig_dir=cfg['paths']['figures_dir'])
sync_to_kaggle_output(p, 'figures')

output_a = {'sweep': sweep, 'stability': stability, 'retest': retest,
            'verdict': verdict, 'config': cfg}
m_path = save_metrics(output_a, 'phase_a', cfg['paths']['metrics_dir'])
sync_to_kaggle_output(m_path, 'metrics')

print('\n' + '='*60)
print(f'GATE A VERDICT: {verdict["gate"]}')
print(f'  Best K: {verdict["best_k"]} (K/D={verdict["best_kd_ratio"]:.2f})')
print(f'  Ground truth: {verdict["ground_truth"]}')
print(f'  Mean importance: {verdict["mean_importance_at_best_k"]:.4f}')
print('  Re-test Spearman:')
for m, s in retest['summary'].items():
    print(f'    {m:<25} {s["spearman_mean"]:+.4f} ± {s["spearman_std"]:.4f}')
print('='*60)

In [ ]:
# Cell 11: GATE A check — stop here if NO-GO
assert verdict['gate'] == 'GO', (
    f'GATE A is NO-GO. Report before proceeding to Phase B.\n{verdict["reasoning"]}')

---
## Phase B — Build the Positive Method
Only runs if GATE A = GO.

In [ ]:
# Cell 12: Phase B stub — implement after GATE A verdict
assert verdict['gate'] == 'GO', 'Phase B requires GATE A = GO'
# TODO: implement Phase B after GATE A report
print('Phase B: TODO')